In [ ]:
import numpy as np
import xarray as xr
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams['axes.unicode_minus'] = False

import cartopy.crs as ccrs


In [ ]:
def ax_pos_inch_to_absolute(fig_size, ax_pos_inch):
    ax_pos_absolute = []
    ax_pos_absolute.append(ax_pos_inch[0]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[1]/fig_size[1])
    ax_pos_absolute.append(ax_pos_inch[2]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[3]/fig_size[1])

    return ax_pos_absolute

In [ ]:
def ax_pos_cm_to_absolute(fig_size, ax_pos_cm):
    ax_pos_absolute = []
    ax_pos_inch = [ pos / 2.54 for pos in ax_pos_cm ]
    ax_pos_absolute.append(ax_pos_inch[0]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[1]/fig_size[1])
    ax_pos_absolute.append(ax_pos_inch[2]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[3]/fig_size[1])
    
    return ax_pos_absolute

In [ ]:
# setting up a custom blue-white-orange color map
from matplotlib.colors import LinearSegmentedColormap

colors = np.array([( 37,  52, 148), ( 44, 127, 184), ( 65, 182, 196), (161, 218, 180), (255, 255, 204),
                   (255, 255, 255),
                   (255, 255, 204), (254, 204,  92), (253, 141,  60), (240,  59,  32), (189,   0,  38)]) / 255.

cmc = LinearSegmentedColormap.from_list('BYWYR', colors, N=101)


In [ ]:
# base dir
base_dir = (Path.cwd() / "../../").resolve()
data_dir = base_dir / "data"
save_dir = base_dir / "figures"

In [ ]:
# filtered data
file_name = "olr-2xdaily-1981-2010-window-360-skip-180-filters.nc"
ds_filters = xr.open_dataset(str(data_dir / file_name))

In [ ]:
# filtered background
file_name = "ou-realization-2024-epsilon0-5.8-lambda0-0.06-tau0-2.3-filters.nc"
ds_filters_background = xr.open_dataset(str(data_dir / file_name))

In [ ]:
# extract grid
lond = ds_filters.lon.values
latd = ds_filters.lat.values

In [ ]:
# variance - filtered background variance
F = [None] * 8
F[0] = np.mean(np.var(ds_filters.Fw1.values, axis=1), axis=0) - np.mean(np.var(ds_filters_background.Fw1, axis=1), axis=0)
F[1] = np.mean(np.var(ds_filters.Fw2.values, axis=1), axis=0) - np.mean(np.var(ds_filters_background.Fw2, axis=1), axis=0)
F[2] = np.mean(np.var(ds_filters.Fw3.values, axis=1), axis=0) - np.mean(np.var(ds_filters_background.Fw3, axis=1), axis=0)
F[3] = np.mean(np.var(ds_filters.Fw4.values, axis=1), axis=0) - np.mean(np.var(ds_filters_background.Fw4, axis=1), axis=0)
F[4] = np.mean(np.var(ds_filters.Fw5.values, axis=1), axis=0) - np.mean(np.var(ds_filters_background.Fw5, axis=1), axis=0)
F[5] = np.mean(np.var(ds_filters.Fw6.values, axis=1), axis=0) - np.mean(np.var(ds_filters_background.Fw6, axis=1), axis=0)
F[6] = np.mean(np.var(ds_filters.Fw7.values, axis=1), axis=0) - np.mean(np.var(ds_filters_background.Fw7, axis=1), axis=0)
F[7] = np.mean(np.var(ds_filters.Fw8.values, axis=1), axis=0) - np.mean(np.var(ds_filters_background.Fw8, axis=1), axis=0)

In [ ]:
# corrections
for i in range(8):

    # compensating for hanning window
    F[i] *= (8. / 3.)
    
    # one fourth of the variance is lost when filtering only positive frequencies
    F[i] *= 4


In [ ]:
# scales
F_scale = [None] * 8

for i in range(8):
    F_scale[i] = np.max(np.abs(F[i]))
    print(F_scale[i].item())


In [ ]:
# clevels
clevels = [None] * 8

clevels[0] = np.linspace(-200, 200, 21)
clevels[1] = np.linspace(-300, 300, 21)
clevels[2] = np.linspace(-200, 200, 21)
clevels[3] = np.linspace(-800, 800, 21)
clevels[4] = np.linspace(-100, 100, 21)
clevels[5] = np.linspace(-100, 100, 21)
clevels[6] = np.linspace(-150, 150, 21)
clevels[7] = np.linspace(-100, 100, 21)

In [ ]:
fig_size = (17.00/2.54, 20.00/2.54)
fig = plt.figure(figsize=fig_size)

ax = []
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.00, 16.00, 07.00, 03.50]), projection=ccrs.PlateCarree(central_longitude=0.0)))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [09.00, 16.00, 07.00, 03.50]), projection=ccrs.PlateCarree(central_longitude=0.0)))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.00, 11.00, 07.00, 03.50]), projection=ccrs.PlateCarree(central_longitude=0.0)))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [09.00, 11.00, 07.00, 03.50]), projection=ccrs.PlateCarree(central_longitude=0.0)))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.00, 06.00, 07.00, 03.50]), projection=ccrs.PlateCarree(central_longitude=0.0)))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [09.00, 06.00, 07.00, 03.50]), projection=ccrs.PlateCarree(central_longitude=0.0)))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.00, 01.00, 07.00, 03.50]), projection=ccrs.PlateCarree(central_longitude=0.0)))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [09.00, 01.00, 07.00, 03.50]), projection=ccrs.PlateCarree(central_longitude=0.0)))

cax = []
cax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.50, 15.50, 06.00, 00.25])))
cax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [09.50, 15.50, 06.00, 00.25])))
cax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.50, 10.50, 06.00, 00.25])))
cax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [09.50, 10.50, 06.00, 00.25])))
cax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.50, 05.50, 06.00, 00.25])))
cax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [09.50, 05.50, 06.00, 00.25])))
cax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.50, 00.50, 06.00, 00.25])))
cax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [09.50, 00.50, 06.00, 00.25])))


panel_name = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']

filter_name = ['Filter 1 (equatorial Kelvin wave)',
               'Filter 2 (Non-disp. eastward branch)',
               'Filter 3 (Non-disp. westward branch)',
               'Filter 4 (central lobe)',
               'Filter 5 (eastward side lobe)',
               'Filter 6 (westward side lobe)',
               'Filter 7 (equatorial MJO)',
               'Filter 8 (equatorial Rossby)']

for i in range(8):

    cs0 = ax[i].contourf(lond, latd, F[i], levels=clevels[i], extend='both', cmap=cmc, transform=ccrs.PlateCarree(central_longitude=0.0))
    cs0 = ax[i].contourf(lond, latd, F[i], levels=clevels[i], extend='both', cmap=cmc, transform=ccrs.PlateCarree(central_longitude=0.0))

    ax[i].coastlines(linewidths=0.5, color='black')

    ax[i].text(00.02, 00.97, panel_name[i],
               ha='left', va='top', transform=ax[i].transAxes, fontsize=10, fontweight='bold', color='black')
    
    ax[i].text(00.50, 01.01, filter_name[i],
               ha='center', va='bottom', transform=ax[i].transAxes, fontsize=10, fontweight='bold', color='black')

    ax[i].gridlines(draw_labels=False, linestyle='--', color='black', alpha=0.2)

    cbar = fig.colorbar(cs0, cax=cax[i], orientation='horizontal', extend='both') #, label=r'$\mathrm{W m^{-2}}$')
    cax[i].set_xlabel(r'[W m$^{-2}$]$\,^{2}$', fontsize=8, va='top', ha='left', color='black')#, labelpad=10)
    
    cax[i].xaxis.set_label_coords(01.05, -00.50, transform=cax[i].transAxes)

    cax[i].tick_params(labelsize=8)
    

In [ ]:
file_name = "fig-s06"
Path(save_dir).mkdir(parents=True, exist_ok=True)
fig.savefig(str(save_dir / file_name) + ".png", dpi=600, format='png', facecolor='white')
fig.savefig(str(save_dir / file_name) + ".pdf", dpi=600, format='pdf', facecolor='white')